<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/portal/_resources/common/imatges/sala_de_premsa/noticies/2016/202-nova-marca-uoc.jpg", align="left" width="380" height="120">

</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">Optimización de bases de datos en entornos analíticos</p>
<p style="margin: 0; text-align:right;">Grado en Ciencia de Datos Aplicada</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudios de Informática, Multimedia y Telecomunicaciones</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

[StackOverflow](https://stackoverflow.com/) es una plataforma popular y de amplia adopción que permite plantear preguntas y respuestas de diferentes temas, como por ejemplo programación. En este conjunto de ejercicios, descargaremos un dataset de StackOverflow y lo analizaremos utilizando ArangoDB. En particular, realizaremos distintas preguntas para entender mejor como interacciona y se comporta la comunidad de StackOverflow. El análisis realizado en estos ejercicios se basa en el siguiente [artículo científico](https://ink.library.smu.edu.sg/cgi/viewcontent.cgi?referer=&httpsredir=1&article=2810&context=sis_research), que recomendamos leer antes de empezar a realizar los ejercicios.  

Esta práctica está basada en la [asignatura BDGE del Máster de Big Data de la UM/USC](https://github.com/dsevilla/bdge/tree/22-23).

# Descarga del dataset

El siguiente comando descarga el dataset. Puede tardar varios minutos.

In [ ]:
!wget https://dn802605.us.archive.org/0/items/stackexchange/es.meta.stackoverflow.com.7z

Descomprimimos el dataset.

In [ ]:
!7z x es.meta.stackoverflow.com.7z

# Información del dataset

El dataset con el que vamos a trabajar contiene 4 archivos serializados en xml.
- `Users.xml`: contiene información de los usuarios.
- `Posts.xml`: contiene información de los posts (preguntas y respuestas).
- `Tags.xml`: contiene información de los tags. Cada post puede tener uno o más tags.
- `Comments.xml`: contiene información de los comentarios. Cada post puede tener uno o más comentarios.

Los siguientes bloques de código se encargan de parsear y organizar los ficheros xml.

In [ ]:
import xmltodict

def parse_xml(path):
    with open(path, 'r', encoding='utf-8') as file:
        my_xml = file.read()
    return xmltodict.parse(my_xml)

In [ ]:
users = parse_xml('Users.xml')['users']['row']
posts = parse_xml('Posts.xml')['posts']['row']
tags = parse_xml('Tags.xml')['tags']['row']
comments = parse_xml('Comments.xml')['comments']['row']

In [ ]:
def remove_first_character_keys(l):
    return [{x[1:]: y for x, y in e.items()} for e in l]
def rename_id(l):
    for e in l:
        e['_key'] = e['Id']
        del e['Id']

users = remove_first_character_keys(users)
posts = remove_first_character_keys(posts)
comments = remove_first_character_keys(comments)
tags = remove_first_character_keys(tags)

rename_id(users)
rename_id(posts)
rename_id(comments)
rename_id(tags)

In [ ]:
for p in posts:
    if 'Tags' in p:
        t = p['Tags']
        t = t[1:-1]
        t = t.split('><')
        p['Tags'] = t

Cuando hayamos ejecutado los bloques anteriores, tendremos cuatro listas de diccionarios (`users`, `posts`, `comments`, `tags`). A continuación se muestra el quinto usuario, post, comentario y tag.

In [ ]:
users[5]

In [ ]:
posts[5]

In [ ]:
comments[5]

In [ ]:
tags[5]

Campos a destacar de un usuario:
- `_key`: identificador único del usuario.

Campos a destacar de un post:
- `_key`: identificador único del post.
- `PostTypeId`: tipo de post. Si es 1 entonces en una pregunta. Si es 2, entonces es una respuesta.
- `tags`: lista de tags.
- `OwnerUserId`: `_key` del usuario que ha creado el post.
- `ParentId`: `_key` del post al que responde. Nótese que este campo no existe si el tipo de post es una pregunta.

Campos a destacar de un comentario:
- `_key`: identificador único del comentario.
- `UserId`: `_key` del usuario que ha creado el comentario.

Campos a destacar de un tag:
- `_key`: identificador único del tag.
- `TagName`: nombre del tag.

# Creación de la base de datos

En primer lugar generamos un objeto Cliente que se conecta a ArangoDB. Luego accedemos a la base de datos `_system` y con ella creamos la base de datos de stackoverflow. Finalmente, instanciamos dicha bbdd en la variable `db`.

In [ ]:
from arango import ArangoClient

client = ArangoClient(hosts="http://arangodb:8529")
sys_db = client.db('_system', username='root', password='rootpassword')
sys_db.create_database('stackoverflow')
db = client.db("stackoverflow", username="root", password="rootpassword")

Ahora creamos el grafo de stackoverflow y lo instanciamos.

In [ ]:
graph = db.create_graph("stackoverflow_graph")

A continuación, creamos las colecciones de vértices que van a formar el grafo (usuarios, posts y tags).

In [ ]:
users_vertices = graph.create_vertex_collection("users")
posts_vertices = graph.create_vertex_collection("posts")
tags_vertices = graph.create_vertex_collection("tags")

Definimos los comentarios como aristas entre los usuarios y los posts.

In [ ]:
comments_edges = graph.create_edge_definition(
    edge_collection="comments",
    from_vertex_collections=["users"],
    to_vertex_collections=["posts"]
)

A continuación insertamos los usuarios, posts y tags en sus respectivas colecciones.

In [ ]:
from tqdm.notebook import tqdm

for user in tqdm(users, desc='Inserting users'):
    users_vertices.insert(user)

for post in tqdm(posts, desc='Inserting posts'):
    posts_vertices.insert(post)

from tqdm.notebook import tqdm

for tag in tqdm(tags, desc='Inserting tags'):
    tags_vertices.insert(tag)

Ahora insertamos los comentarios en la colección de aristas ignorando aquellos comentarios que no tienen creador. Nótese que definimos los campos `_from` y `_to` usando los `_ids` (concatenación del nombre de la colección y la `_key`) de los usuarios y los posts.

In [ ]:
for comment in tqdm(comments, desc='Inserting comments'):
    comment_new = comment.copy()
    if 'UserId' not in comment_new:
        continue
    comment_new['_from'] = "users/" + comment_new['UserId']
    comment_new['_to'] = "posts/" + comment_new['PostId']
    comments_edges.insert(comment_new)

Ahora creamos la colección de aristas `owner_relation`. Estas aristas van de posts a usuarios y significa "escrito por".

In [ ]:
owner_edges = graph.create_edge_definition(
    edge_collection="owner_relation",
    from_vertex_collections=["posts"],
    to_vertex_collections=["users"]
)

Insertamos entonces la aristas recorreindo los posts y usando el campo `OwnerUserId`.

In [ ]:
for post in tqdm(posts, desc='Inserting owner edges'):
    if 'OwnerUserId' not in post:
        continue
    edge = {'_to': "users/" + post['OwnerUserId'],
           '_from': "posts/" + post['_key']}
    owner_edges.insert(edge)

Para completar nuestro grafo falta definir las aristas que relacionan los posts con los tags. Eso lo haremos en el siguiente apartado para ilustrar AQL.

# ArangoDB Query Language (AQL)

En este apartado mostramos varios ejemplos de consultas a ArangoDB usando AQL. En primer lugar instanciamos el motor AQL.

In [ ]:
aql = db.aql

Estamos interesados en, para cada post, retornar su identificador y sus tags (solo si tiene) en orden ascendente con respecto a su identificador.

In [ ]:
# For each post return its tags (if the post has tags)
aql_query = '''
FOR post in posts
    FILTER post["Tags"] != null
    SORT post["_key"] ASC
    RETURN {id: post["_key"], tags: post["Tags"]}
'''
# Execute the query
cursor = db.aql.execute(
  aql_query
)
# Iterate through the result cursor
ids_tags = [doc for doc in cursor]
ids_tags[0:5]

A modo de ilustración, vamos a crear la relación `tagged` que conecta los posts con los tags usando AQL. Antes ejecutar dicha consulta, creamos un índice hash en el campo `TagName` de los tags para acelerar la consulta.

In [ ]:
tags_vertices.add_hash_index(fields=['TagName'], unique=True)

Creamos ahora la colección de aristas.

In [ ]:
# link the posts with the tags
tags_edges = graph.create_edge_definition(
    edge_collection="tagged",
    from_vertex_collections=["posts"],
    to_vertex_collections=["tags"]
)

Y ejecutamos el siguiente bloque AQL. Para cada post que tenga tags, recorre sus tags y haz un "join" con la colección tags para recuperar el `_id` y poder insertar las aristas en la colección tagged.

In [ ]:
aql_query = '''
FOR post in posts
    FILTER post["Tags"] != null
    FOR tag_post in post["Tags"]
        FOR tag in tags
            FILTER tag["TagName"] == tag_post
            INSERT {"_from": post._id, "_to": tag._id} IN tagged
'''
cursor = aql.execute(
  aql_query
)

Ahora podemos realizar la consula de los posts de cada tag usando los "graph traversals" de AQL. esta consulta debería ser equivalente a la primera que hicimos.

In [ ]:
aql_query = '''
FOR post in posts
    LET post_tags = (
        FOR t in 1..1 OUTBOUND post tagged
        RETURN t["TagName"]
        )
    FILTER LENGTH(post_tags) > 0
    SORT post["_key"] ASC
    RETURN {id: post["_key"], tags: post_tags}
'''
cursor = db.aql.execute(
  aql_query
)
ids_tags = [doc for doc in cursor]
ids_tags[0:5]

# Ejercicios

Si ya habeís ejecutado los apartados anteriores, no es neceario volverlos a ejecutar y podéis empezar directamente desde aquí.

In [ ]:
from arango import ArangoClient

client = ArangoClient(hosts="http://arangodb:8529")
db = client.db("stackoverflow", username="root", password="rootpassword")
aql = db.aql

Ahora vamos a replicar los resultados de este [artículo científico](https://ink.library.smu.edu.sg/cgi/viewcontent.cgi?referer=&httpsredir=1&article=2810&context=sis_research) pero con nuestro dataset. Cada ejercicio replica una pregunta de investigación. Vamos a replicar las cuatro primeras usando AQL.

## RQ1: Distribución de interrogadores

Para cada usuario calcula el número de preguntas que ha escrito (usa graph traversals).

Para cada usuario calcula el número de preguntas que ha escrito (usa la colección de aristas directamente).

## RQ2: Distribución de respuestas por usuario

Para cada usuario calcula el número de respuestas que ha escrito (usa graph traversals).

Para cada usuario calcula el número de preguntas que ha escrito (usa la colección de aristas directamente).

## RQ3: Segregación de la comunidad StackOverflow

Para cada usuario calcular en número de posts que ha escrito y el número de respuestas. Devuelve un diccionario que contenga el `_id` del usuario, el número de posts (preguntas y respuestas) y el número de respuestas que ha escrito. Usa graph traversals.

## RQ4: Reciprocidad en StackOverflow

Para responder a esta pregunta de investigación, vamos a llevar a cabo los dos siguientes pasos:

1. Creación de las aristas `help`. Es decir, vamos a crear el grafo de ayuda del cual habla el artículo. Un usuario _u1_ ayuda a otro usuario _u2_ cuando _u1_ ha escrito una respuesta a una pregunta hecha por _u2_.

2. Crearemos la consulta final que devuelva los pares recíprocos. Dos usuarios son recíprocos si se han ayudado mutuamente.

Para hacer este ejercicio, ten en cuenta que ArangoDB maneja multigrafos (es decir, pueden haber múltiples aristas de un nodo a otro).

Creamos la colección `help` edges (si la colección ya la has creado, sáltatelo).

Para cada post, filtra los posts que no son respuestas y no tienen autor y de ahí obtienes el usuario que ayuda. Luego haz un "join" con los posts para localizar la pregunta y el usuario que es ayudado. Finalmente inserta la arista en la colección help. Recuerda que los atributos `_from` e `_to` necesita de `_id`. Por tanto, tendrás que usar la función `CONCAT` para concatenar el nombre de la colección con los identificadores.

Haz una query que te devuelva, para cada usuario, sus recíprocos. Un usuario es recíproco de otro si se ayudan mutuamente.